# Notebook for exporting model to onnx and performing quantisation
The goal is to produce a model that can be deployed to edge devices like Raspberry Pi, Jetson etc.  
Connects to the theme that a driver distraction classifier model might be run locally in a car from a camera/cameras in the cabin. 
Like the driver alertness monitoring seen in many modern cars. 

In [2]:
import torch
import torch.nn as nn
import onnx
import onnxruntime as ort
import numpy as np
from pathlib import Path
import time
import sys

sys.path.append("../..")
from model.projectmodels import get_model

### Config

In [33]:
model_name = "minicnn"
num_classes = 10
ckpt_path = Path(f"../checkpoints/{model_name}_best.pth")
onnx_path = Path(f"../onnx/{model_name}.onnx")
onnx_p_path = Path(f"../onnx/{model_name}_p.onnx")
onnx_q_path = Path(f"../onnx/{model_name}_int8.onnx")

device = "cuda" if torch.cuda.is_available() else "cpu"
onnx_path.parent.mkdir(parents=True, exist_ok=True)

### Load trained PyTorch model

In [24]:
model = get_model(model_name, num_classes=num_classes)
state = torch.load(ckpt_path, map_location="cpu")
model.load_state_dict(state, strict=True)
model.eval().to(device)

dummy_input = torch.randn(1, 3, 224, 224).to(device) # just need the right shape

### Export to ONNX

In [25]:
torch.onnx.export(
    model,
    dummy_input,
    onnx_path.as_posix(),
    opset_version=18,
    input_names=["input"],
    output_names=["logits"],
    dynamic_axes=None,
    do_constant_folding=True,
    export_params=True,
    external_data=False
)

print("Exported ONNX to:", onnx_path)
onnx_model = onnx.load(onnx_path.as_posix())
onnx.checker.check_model(onnx_model)
print("ONNX model is valid.")

[torch.onnx] Obtain model graph for `MiniCNN([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `MiniCNN([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Exported ONNX to: ..\onnx\minicnn.onnx
ONNX model is valid.


C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.2800.0_x64__qbz5n2kfra8p0\Lib\copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


### Baseline ONNX Inference + latency

In [32]:
sess = ort.InferenceSession(
    onnx_path.as_posix(),
    providers=["CPUExecutionProvider"]
)

def ort_infer(session, x_np):
    inputs = {session.get_inputs()[0].name: x_np}
    return session.run(None, inputs)[0]

# warmup
x = dummy_input.cpu().numpy()
for _ in range(10):
    _ = ort_infer(sess, x)

# measure latency
runs = 50
start = time.time()
for _ in range(runs):
    _ = ort_infer(sess, x)
end = time.time()
print(f"ONNX FP32 latency: {(end - start) / runs * 1000:.2f} ms")

ONNX FP32 latency: 0.26 ms


### Compare PyTorch vs ONNX Outputs

In [27]:
with torch.no_grad():
    torch_out = model(dummy_input).cpu().numpy()

onnx_out = ort_infer(sess, dummy_input.cpu().numpy())

max_abs_diff = np.max(np.abs(torch_out - onnx_out))
print("Max abs diff (PyTorch vs ONNX):", max_abs_diff)

Max abs diff (PyTorch vs ONNX): 0.027387619


### Preprocess ONNX

In [35]:
from onnxruntime.quantization import quant_pre_process

quant_pre_process(
    input_model=onnx_path,
    output_model_path=onnx_p_path
)

### Prepare calibration reader for ONNX

In [37]:
m = onnx.load(onnx_p_path)
print([i.name for i in m.graph.input])

['input']


In [42]:
from onnxruntime.quantization.calibrate import CalibrationDataReader
from model.dataset import get_transforms
import random
from PIL import Image

class MyDataReader(CalibrationDataReader):
    def __init__(self, folder, input_name, num_samples=200):
        self.input_name = input_name
        self.folder = Path(folder)
        
        all_imgs = list(self.folder.glob("*"))
        self.imgs = random.sample(all_imgs, num_samples)

        _, self.transform = get_transforms()

        self.enum_data = iter(self._load_images())

    def _load_images(self):
        for img_path in self.imgs:
            img = Image.open(img_path).convert("RGB")
            x = self.transform(img).unsqueeze(0).numpy()
            yield {self.input_name: x}

    def get_next(self):
        return next(self.enum_data, None)

# Example: list of input dicts
reader = MyDataReader(folder="../data/imgs/test/", input_name="input", num_samples=200)


### Quantise ONNX model using static quantisation (good for CNN)

In [43]:
from onnxruntime.quantization import quantize_static, QuantType, QuantFormat

quantize_static(
    model_input=onnx_p_path.as_posix(),
    model_output=onnx_q_path.as_posix(),
    calibration_data_reader=reader,
    quant_format=QuantFormat.QDQ,
    activation_type=QuantType.QInt8,
    weight_type=QuantType.QInt8,
)

print("Quantized model saved to:", onnx_q_path)

Quantized model saved to: ..\onnx\minicnn_int8.onnx


### Run inference with quantised model

In [44]:
sess_q = ort.InferenceSession(
    onnx_q_path.as_posix(),
    providers=["CPUExecutionProvider"]  # quantized usually CPU-only
)

# warmup
for _ in range(10):
    _ = ort_infer(sess_q, x)

runs = 50
start = time.time()
for _ in range(runs):
    _ = ort_infer(sess_q, x)
end = time.time()
print(f"ONNX INT8 latency: {(end - start) / runs * 1000:.2f} ms")

ONNX INT8 latency: 0.32 ms


### Compare FP32 vs INT8 outputs

In [45]:
onnx_q_out = ort_infer(sess_q, x)
max_abs_diff_q = np.max(np.abs(onnx_out - onnx_q_out))
print("Max abs diff (ONNX FP32 vs ONNX INT8):", max_abs_diff_q)

Max abs diff (ONNX FP32 vs ONNX INT8): 9.937738


### Accuracy check on small validation subset

In [48]:
from torch.utils.data import DataLoader
from model.dataset import DriverDataset
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

val_ds = DriverDataset("../data/imgs/train", transform=transform)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False)

def eval_onnx(session, loader):
    correct = 0
    total = 0
    for imgs, labels in loader:
        x_np = imgs.numpy()
        logits = ort_infer(session, x_np)
        preds = logits.argmax(axis=1)
        correct += (preds == labels.numpy()).sum().item()
        total += labels.size(0)
    return correct / total

acc_fp32 = eval_onnx(sess, val_loader)
acc_int8 = eval_onnx(sess_q, val_loader)

print("ONNX FP32 accuracy:", acc_fp32)
print("ONNX INT8 accuracy:", acc_int8)


ONNX FP32 accuracy: 0.9931769532643596
ONNX INT8 accuracy: 0.9900998929718159
